<a href="https://colab.research.google.com/github/imrangis/imrangis/blob/main/MODIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# MODIS 500 m annual dry-season composites for Sundarbans
# Matched to GEDI L4A plot locations
# ============================================================

# Install / import
!pip install geemap geopandas -q

import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

# Authenticate + initialize
ee.Authenticate()
ee.Initialize(project='esagb-493105')

# Mount drive for reading the GEDI plots
drive.mount('/content/drive')

# ── 1. Study area & GEDI plots ───────────────────────────────
sundarbans = ee.FeatureCollection('projects/esagb-493105/assets/sundarbans')
studyArea  = sundarbans.geometry()

# Load previously-exported thinned GEDI plots
df = pd.read_csv('/content/drive/MyDrive/gedi_plots_thinned.csv')
print(f'Loaded {len(df)} thinned GEDI plots')

# ── 2. Configuration ─────────────────────────────────────────
startYear = 2000       # MODIS Terra launched late 1999; 2000 is safe
endYear   = 2024
CLOUD_THRESH = 20      # % cloud cover allowed per tile (MOD09A1 is pre-composited so clean)

# MODIS band names → friendlier names (500 m surface reflectance)
MODIS_BANDS = ['sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03',
               'sur_refl_b04', 'sur_refl_b06', 'sur_refl_b07']
RENAMED     = ['Red', 'NIR', 'Blue', 'Green', 'SWIR1', 'SWIR2']
# Note: MOD09A1 band order is Red(b1), NIR(b2), Blue(b3), Green(b4),
#       SWIR1(b6, 1628nm), SWIR2(b7, 2105nm). Band 5 is 1240nm (less common, skipped).

# GEDI-matching resolution for final output
TARGET_SCALE = 25

# ── 3. MODIS cloud/quality masking ───────────────────────────
# MOD09A1 has a StateQA band with cloud, cirrus, aerosol, snow bits.
def maskMODISQuality(image):
    qa = image.select('StateQA')
    # Bit 0-1: cloud state (0 = clear)
    cloud  = qa.bitwiseAnd(3).eq(0)
    # Bit 2: cloud shadow (0 = no)
    shadow = qa.bitwiseAnd(1 << 2).eq(0)
    # Bit 8-9: cirrus (0 = none)
    cirrus = qa.bitwiseAnd(3 << 8).eq(0)
    # Bit 10: internal cloud algorithm flag (0 = no cloud)
    icf    = qa.bitwiseAnd(1 << 10).eq(0)
    # Bit 15: snow/ice flag (0 = no)
    snow   = qa.bitwiseAnd(1 << 15).eq(0)
    mask = cloud.And(shadow).And(cirrus).And(icf).And(snow)
    return (image
            .updateMask(mask)
            .multiply(0.0001)      # MOD09A1 scale factor for reflectance
            .copyProperties(image, ['system:time_start']))

# ── 4. Rename bands ───────────────────────────────────────────
def renameMODIS(image):
    return image.select(MODIS_BANDS, RENAMED)

# ── 5. Load MODIS MOD09A1 (Terra, 500 m, 8-day) ──────────────
modisAll = (ee.ImageCollection('MODIS/061/MOD09A1')
            .filterDate(f'{startYear}-01-01', f'{endYear}-12-31')
            .filterBounds(studyArea)
            .map(maskMODISQuality)
            .map(renameMODIS)
            .map(lambda img: img.clip(studyArea)))

print('Total MODIS MOD09A1 8-day images:', modisAll.size().getInfo())

# ── 6. Build annual dry-season composites ────────────────────
# Sundarbans dry season: Dec (prev year) through Feb (current year)
def makeAnnualComposite(year):
    year = ee.Number(year)
    seasonStart = ee.Date.fromYMD(year.subtract(1), 12, 1)
    seasonEnd   = ee.Date.fromYMD(year, 2, 28)

    seasonal = modisAll.filterDate(seasonStart, seasonEnd)
    count    = seasonal.size()

    # Median composite — robust to residual cloud/haze
    composite = ee.Image(ee.Algorithms.If(
        count.gt(0),
        seasonal.median().select(RENAMED),
        ee.Image.constant([0, 0, 0, 0, 0, 0])
            .rename(RENAMED)
            .toFloat()
            .updateMask(ee.Image.constant(0))
    ))

    return composite.set({
        'system:time_start': ee.Date.fromYMD(year, 1, 15).millis(),
        'year':              year,
        'system:index':      year.format('%d'),
        'image_count':       count
    })

years = ee.List.sequence(startYear, endYear)
annualModis = ee.ImageCollection.fromImages(
    years.map(lambda y: makeAnnualComposite(y))
)

print('Annual MODIS composites built:', annualModis.size().getInfo())

# ── 7. Quick diagnostic: images per dry season year ──────────
def count_season(year):
    year = ee.Number(year)
    seasonStart = ee.Date.fromYMD(year.subtract(1), 12, 1)
    seasonEnd   = ee.Date.fromYMD(year, 2, 28)
    n = modisAll.filterDate(seasonStart, seasonEnd).size()
    return ee.Feature(None, {'year': year, 'count': n})

counts = ee.FeatureCollection(years.map(lambda y: count_season(y)))
count_df = pd.DataFrame([f['properties'] for f in counts.getInfo()['features']])
print('\nMODIS 8-day images per dry season:')
print(count_df.to_string(index=False))


In [ ]:
# ============================================================
# Vegetation indices on MODIS annual dry-season composites
# Following Qin et al. / Dubayah et al. recipe
# ============================================================

# ── 10. Add vegetation indices to each annual composite ──────
def addIndices(image):
    """
    Adds 9 vegetation indices:
      NDVI, EVI, ARVI, NDMI, SAVI, TCB, TCG, TCW
    Plus ties everything to the input composite.
    """

    # --- Basic ratios ---
    # NDVI = (NIR - Red) / (NIR + Red)                    Rouse 1974
    ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')

    # NDMI = (NIR - SWIR1) / (NIR + SWIR1)                Wilson & Sader 2002
    ndmi = image.normalizedDifference(['NIR', 'SWIR1']).rename('NDMI')

    # --- Expressions that need multiple bands ---
    # EVI = 2.5 * (NIR - Red) / (NIR + 6*Red - 7.5*Blue + 1)     Huete 2002
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR':  image.select('NIR'),
            'RED':  image.select('Red'),
            'BLUE': image.select('Blue')
        }
    ).rename('EVI')

    # ARVI = (NIR - (2*Red - Blue)) / (NIR + (2*Red - Blue))     Kaufman & Tanre 1992
    arvi = image.expression(
        '(NIR - (2*RED - BLUE)) / (NIR + (2*RED - BLUE))',
        {
            'NIR':  image.select('NIR'),
            'RED':  image.select('Red'),
            'BLUE': image.select('Blue')
        }
    ).rename('ARVI')

    # SAVI = ((NIR - Red) / (NIR + Red + L)) * (1 + L),   L = 0.5     Huete 1988
    savi = image.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
        {
            'NIR': image.select('NIR'),
            'RED': image.select('Red')
        }
    ).rename('SAVI')

    # --- Tasseled Cap Transformation (MODIS coefficients) ---
    # Lobser & Cohen 2007 — 6-band MODIS coefficients
    # Order: Red(b1), NIR(b2), Blue(b3), Green(b4), SWIR1(b6), SWIR2(b7)
    # But we'll use our renamed bands in a consistent order.

    # TCB (Brightness)
    tcb = image.expression(
        '0.4395*RED + 0.5945*NIR + 0.2460*BLUE + 0.3918*GREEN + 0.3506*SWIR1 + 0.2136*SWIR2',
        {
            'RED':   image.select('Red'),
            'NIR':   image.select('NIR'),
            'BLUE':  image.select('Blue'),
            'GREEN': image.select('Green'),
            'SWIR1': image.select('SWIR1'),
            'SWIR2': image.select('SWIR2')
        }
    ).rename('TCB')

    # TCG (Greenness)
    tcg = image.expression(
        '-0.4064*RED + 0.5129*NIR - 0.2744*BLUE - 0.2893*GREEN + 0.4882*SWIR1 - 0.0036*SWIR2',
        {
            'RED':   image.select('Red'),
            'NIR':   image.select('NIR'),
            'BLUE':  image.select('Blue'),
            'GREEN': image.select('Green'),
            'SWIR1': image.select('SWIR1'),
            'SWIR2': image.select('SWIR2')
        }
    ).rename('TCG')

    # TCW (Wetness)
    tcw = image.expression(
        '0.1147*RED + 0.2489*NIR + 0.2408*BLUE + 0.3132*GREEN - 0.3122*SWIR1 - 0.6416*SWIR2',
        {
            'RED':   image.select('Red'),
            'NIR':   image.select('NIR'),
            'BLUE':  image.select('Blue'),
            'GREEN': image.select('Green'),
            'SWIR1': image.select('SWIR1'),
            'SWIR2': image.select('SWIR2')
        }
    ).rename('TCW')

    # Stack everything
    return (image
            .addBands([ndvi, evi, arvi, ndmi, savi, tcb, tcg, tcw])
            .toFloat()
            .copyProperties(image, ['system:time_start', 'year',
                                    'system:index', 'image_count']))

# Apply to the annual composites
annualWithIndices = annualModis.map(addIndices)

# Quick check: bands in the first composite
print('Bands in annual composite with indices:')
print(annualWithIndices.first().bandNames().getInfo())

# ── 11. Pick target year and inspect ─────────────────────────
targetYear = 2020

composite_2020 = (annualWithIndices
                  .filter(ee.Filter.eq('year', targetYear))
                  .first())

print(f'\n{targetYear} composite band list:')
print(composite_2020.bandNames().getInfo())

# ── 12. Visualize a few indices ──────────────────────────────
Map = geemap.Map()
Map.centerObject(studyArea, 10)

Map.addLayer(composite_2020.select('NDVI').clip(studyArea),
             {'min': 0, 'max': 0.9,
              'palette': ['white', 'yellow', 'green', 'darkgreen']},
             f'NDVI {targetYear}', True)

Map.addLayer(composite_2020.select('EVI').clip(studyArea),
             {'min': 0, 'max': 0.6,
              'palette': ['white', 'lightgreen', 'green', 'darkgreen']},
             f'EVI {targetYear}', False)

Map.addLayer(composite_2020.select('NDMI').clip(studyArea),
             {'min': -0.3, 'max': 0.5,
              'palette': ['brown', 'white', 'blue']},
             f'NDMI {targetYear} (moisture)', False)

Map.addLayer(composite_2020.select('TCG').clip(studyArea),
             {'min': -0.1, 'max': 0.4,
              'palette': ['brown', 'yellow', 'green']},
             f'TCG {targetYear}', False)

Map.addLayer(composite_2020.select('TCW').clip(studyArea),
             {'min': -0.5, 'max': 0.2,
              'palette': ['red', 'white', 'blue']},
             f'TCW {targetYear} (wetness)', False)

Map.addLayer(studyArea, {'color': 'white'}, 'Boundary')
Map

Bands in annual composite with indices:
['Red', 'NIR', 'Blue', 'Green', 'SWIR1', 'SWIR2', 'NDVI', 'EVI', 'ARVI', 'NDMI', 'SAVI', 'TCB', 'TCG', 'TCW']

2020 composite band list:
['Red', 'NIR', 'Blue', 'Green', 'SWIR1', 'SWIR2', 'NDVI', 'EVI', 'ARVI', 'NDMI', 'SAVI', 'TCB', 'TCG', 'TCW']


Map(center=[22.067483656354554, 89.48116429047039], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Load the uploaded asset
plots_ee = ee.FeatureCollection('projects/esagb-493105/assets/gedi_plots_thinned')
print(f'Plots loaded from asset: {plots_ee.size().getInfo()}')
print('Properties:', plots_ee.first().propertyNames().getInfo())

In [ ]:
# ── 13. Load plots from GEE Asset ────────────────────────────
plots_ee = ee.FeatureCollection('projects/esagb-493105/assets/gedi_plots_thinned')

print(f'Plots in asset: {plots_ee.size().getInfo()}')
print('Properties in asset:', plots_ee.first().propertyNames().getInfo())
# Should show: ['agbd', 'agbd_se', 'x', 'y', ...] — x, y already per-plot

# ── 13b. Add WGS84 lon/lat for each plot ─────────────────────
# Each plot has its own geometry, so this computes lon/lat per plot
def addCoords(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'lon': ee.Number(coords.get(0)),
        'lat': ee.Number(coords.get(1))
    })

plots_ee = plots_ee.map(addCoords)

# Verify: every feature now has lon, lat, plus original x, y
print('Properties after addCoords:', plots_ee.first().propertyNames().getInfo())

# ── 14. Sample MODIS predictors at each plot ─────────────────
targetYear = 2020

composite_target = (annualWithIndices
                    .filter(ee.Filter.eq('year', targetYear))
                    .first())

PREDICTORS = ['Red', 'NIR', 'Blue', 'Green', 'SWIR1', 'SWIR2',
              'NDVI', 'EVI', 'ARVI', 'NDMI', 'SAVI',
              'TCB', 'TCG', 'TCW']

composite_25m = (composite_target
                 .select(PREDICTORS)
                 .resample('bilinear')
                 .reproject(crs='EPSG:32645', scale=25))

# sampleRegions preserves ALL input properties (agbd, agbd_se, x, y, lon, lat)
# AND adds all 14 predictor bands per plot
sampled = composite_25m.sampleRegions(
    collection=plots_ee,
    scale=25,
    geometries=True,
    tileScale=4
)

# Verify what will be in the CSV
print('Final sampled feature properties:')
print(sampled.first().propertyNames().getInfo())
# Expected: ['agbd', 'agbd_se', 'x', 'y', 'lon', 'lat',
#            'Red', 'NIR', 'Blue', 'Green', 'SWIR1', 'SWIR2',
#            'NDVI', 'EVI', 'ARVI', 'NDMI', 'SAVI',
#            'TCB', 'TCG', 'TCW']

# ── 15. Export to Drive with explicit columns ────────────────
task = ee.batch.Export.table.toDrive(
    collection=sampled,
    description=f'GEDI_MODIS_predictors_{targetYear}',
    folder='GEE_exports',
    fileNamePrefix=f'gedi_modis_predictors_{targetYear}',
    fileFormat='CSV',
    selectors=[
        # Response variables
        'agbd', 'agbd_se',
        # Coordinates (per plot)
        'x', 'y',           # UTM 45N (meters)
        'lon', 'lat',       # WGS84 (degrees)
        # Spectral bands
        'Red', 'NIR', 'Blue', 'Green', 'SWIR1', 'SWIR2',
        # Vegetation indices
        'NDVI', 'EVI', 'ARVI', 'NDMI', 'SAVI',
        'TCB', 'TCG', 'TCW'
    ]
)
task.start()
print('Export started.')

# ── 16. Monitor ──────────────────────────────────────────────
import time
while task.active():
    print(f'  Status: {task.status()["state"]}')
    time.sleep(30)
print('Done:', task.status())

Plots in asset: 76988
Properties in asset: ['agbd', 'agbd_se', 'system:index']
Properties after addCoords: ['lat', 'lon', 'agbd', 'agbd_se', 'system:index']
Final sampled feature properties:
['agbd', 'lon', 'agbd_se', 'lat', 'Blue', 'SAVI', 'TCW', 'NDVI', 'Red', 'TCB', 'NIR', 'NDMI', 'EVI', 'TCG', 'ARVI', 'Green', 'SWIR1', 'system:index', 'SWIR2']
Export started.
  Status: READY
  Status: READY
  Status: READY
  Status: READY
  Status: READY
  Status: READY
  Status: READY
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
Done: {'state': 'COMPLETED', 'description': 'GEDI_MODIS_predictors_2020', 'priority': 100, 'creation_timestamp_ms': 1776567198677, 'update_timestamp_ms': 1776567756349, 'start_timestamp_ms': 1776567404010, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1vbEsLd94tcs

In [ ]:
# ============================================================
# Load extracted MODIS predictors + GEDI AGBD training table
# ============================================================

import pandas as pd
import numpy as np

# ── 17. Load the CSV from Drive ──────────────────────────────
# Drive should already be mounted from earlier. If not:
# from google.colab import drive
# drive.mount('/content/drive')

csv_path = '/content/drive/MyDrive/GEE_exports/gedi_modis_predictors_2020.csv'
training = pd.read_csv(csv_path)

print(f'Loaded training table: {training.shape[0]} rows × {training.shape[1]} columns')
print(f'\nColumns: {training.columns.tolist()}')
print(f'\nFirst 5 rows:')
training.head()

Loaded training table: 76886 rows × 18 columns

Columns: ['system:index', 'ARVI', 'Blue', 'EVI', 'Green', 'NDMI', 'NDVI', 'NIR', 'Red', 'SAVI', 'SWIR1', 'SWIR2', 'TCB', 'TCG', 'TCW', 'agbd', 'agbd_se', '.geo']

First 5 rows:


,system:index,ARVI,Blue,EVI,Green,NDMI,NDVI,NIR,Red,SAVI,SWIR1,SWIR2,TCB,TCG,TCW,agbd,agbd_se,.geo
0,00000000000000000000_0,0.668449,0.0278,0.433196,0.0515,0.443658,0.736454,0.2652,0.04025,0.418927,0.1022,0.0333,0.245312,0.146910,0.040177,22.312935,2.981796,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,00000000000000000001_0,0.540984,0.0372,0.349570,0.0616,0.334299,0.626412,0.2303,0.05290,0.339760,0.1149,0.0498,0.244370,0.124509,0.023817,21.897257,2.981802,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,00000000000000000002_0,0.512987,0.0358,0.342355,0.0596,0.181242,0.615811,0.2330,0.05540,0.337900,0.1615,0.0821,0.269183,0.148474,-0.011460,23.277212,2.981807,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,00000000000000000003_0,0.737781,0.0287,0.456686,0.0488,0.423701,0.769929,0.2631,0.03420,0.430641,0.1065,0.0330,0.242012,0.150926,0.037181,23.460205,2.981813,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,00000000000000000006_0,0.687257,0.0275,0.406802,0.0484,0.440193,0.738814,0.2390,0.03590,0.393147,0.0929,0.0293,0.222421,0.131694,0.037583,20.558065,2.981876,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [ ]:
# ============================================================
# Random Forest biomass model — hyperparameter tuning
# with 10-fold cross-validation + final test evaluation
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import time

# ── 1. Load the exported CSV ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/GEE_exports/gedi_modis_predictors_2020.csv')
print(f'Loaded {len(df)} rows')

# ── 2. Clean data ────────────────────────────────────────────
PREDICTORS = ['Red', 'NIR', 'Blue', 'Green', 'SWIR1', 'SWIR2',
              'NDVI', 'EVI', 'ARVI', 'NDMI', 'SAVI',
              'TCB', 'TCG', 'TCW']
RESPONSE = 'agbd'

df_clean = df.dropna(subset=PREDICTORS + [RESPONSE]).reset_index(drop=True)
print(f'After dropping NaN: {len(df_clean)} rows')

X = df_clean[PREDICTORS].values
y = df_clean[RESPONSE].values

# ── 3. Split into training (80%) and test (20%) ──────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)
print(f'Training: {X_train.shape[0]} plots | Test: {X_test.shape[0]} plots')

# ── 4. Hyperparameter tuning with 10-fold CV ─────────────────
print('\n' + '='*60)
print('GridSearchCV — 10-fold CV with multiple metrics')
print('='*60)

param_grid = {
    'n_estimators':      [300, 500],
    'max_depth':         [None, 15, 25],
    'min_samples_split': [2, 5],
    'max_features':      ['sqrt', 0.33]
}

total_combos = np.prod([len(v) for v in param_grid.values()])
print(f'Parameter combos: {total_combos} × 10 folds = {total_combos * 10} fits\n')

grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=KFold(n_splits=10, shuffle=True, random_state=42),
    scoring={'r2':   'r2',
             'rmse': 'neg_root_mean_squared_error',
             'mae':  'neg_mean_absolute_error'},
    refit='r2',              # select best by R²
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid.fit(X_train, y_train)
print(f'\nGrid search complete in {(time.time()-start)/60:.1f} minutes')

# rf_best is the final trained model — ready to use
rf_best = grid.best_estimator_

# ── 5. Best hyperparameters and CV metrics ───────────────────
print('\n' + '='*60)
print('Best hyperparameters')
print('='*60)
for param, value in grid.best_params_.items():
    print(f'  {param:20s}: {value}')

# Extract per-fold metrics for the best model from grid.cv_results_
best_idx = grid.best_index_
cv_results = grid.cv_results_

r2_folds   = np.array([cv_results[f'split{i}_test_r2'][best_idx]   for i in range(10)])
rmse_folds = np.array([-cv_results[f'split{i}_test_rmse'][best_idx] for i in range(10)])
mae_folds  = np.array([-cv_results[f'split{i}_test_mae'][best_idx]  for i in range(10)])

print(f'\n10-fold CV metrics (best model):')
print(f'  R²   = {r2_folds.mean():.3f} ± {r2_folds.std():.3f}')
print(f'  RMSE = {rmse_folds.mean():.2f} ± {rmse_folds.std():.2f} Mg/ha')
print(f'  MAE  = {mae_folds.mean():.2f} ± {mae_folds.std():.2f} Mg/ha')

# ── 6. Evaluate on held-out test set ─────────────────────────
print('\n' + '='*60)
print('Held-out test set evaluation')
print('='*60)

y_pred_test = rf_best.predict(X_test)

test_r2   = r2_score(y_test, y_pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_mae  = mean_absolute_error(y_test, y_pred_test)
test_bias = np.mean(y_pred_test - y_test)

print(f'  R²   = {test_r2:.3f}')
print(f'  RMSE = {test_rmse:.2f} Mg/ha')
print(f'  MAE  = {test_mae:.2f} Mg/ha')
print(f'  Bias = {test_bias:+.2f} Mg/ha')

# ── 7. Feature importance ────────────────────────────────────
importances = pd.DataFrame({
    'Feature': PREDICTORS,
    'Importance': rf_best.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('\nFeature importance ranking:')
print(importances.to_string(index=False))

# ── 8. Diagnostic plots ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Predicted vs observed
ax = axes[0, 0]
ax.scatter(y_test, y_pred_test, alpha=0.3, s=8, c='darkgreen')
lims = [0, max(y_test.max(), y_pred_test.max())]
ax.plot(lims, lims, 'r--', lw=1, label='1:1 line')
ax.set_xlabel('Observed AGBD (Mg/ha)')
ax.set_ylabel('Predicted AGBD (Mg/ha)')
ax.set_title(f'Test set: Predicted vs Observed\n'
             f'R² = {test_r2:.3f}, RMSE = {test_rmse:.1f} Mg/ha')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(lims); ax.set_ylim(lims)

# Residuals
ax = axes[0, 1]
residuals = y_pred_test - y_test
ax.scatter(y_test, residuals, alpha=0.3, s=8, c='steelblue')
ax.axhline(0, color='r', linestyle='--', lw=1)
ax.set_xlabel('Observed AGBD (Mg/ha)')
ax.set_ylabel('Residual (Pred - Obs, Mg/ha)')
ax.set_title(f'Residuals\nBias = {test_bias:+.2f} Mg/ha')
ax.grid(alpha=0.3)

# Feature importance
ax = axes[1, 0]
ax.barh(importances['Feature'], importances['Importance'], color='darkgreen')
ax.set_xlabel('Importance')
ax.set_title('Feature importance')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# Residual distribution
ax = axes[1, 1]
ax.hist(residuals, bins=50, color='steelblue', edgecolor='white')
ax.axvline(0, color='r', linestyle='--', lw=1)
ax.set_xlabel('Residual (Pred - Obs, Mg/ha)')
ax.set_ylabel('Count')
ax.set_title(f'Residual distribution\n'
             f'Mean = {residuals.mean():+.2f}, StdDev = {residuals.std():.2f}')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── 9. Save model ────────────────────────────────────────────
import joblib
model_path = '/content/drive/MyDrive/GEE_exports/rf_biomass_model_2020.pkl'
joblib.dump(rf_best, model_path)
print(f'\nModel saved to: {model_path}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 76886 rows
After dropping NaN: 76886 rows
Training: 61508 plots | Test: 15378 plots

GridSearchCV — 10-fold CV with multiple metrics
Parameter combos: 24 × 10 folds = 240 fits

Fitting 10 folds for each of 24 candidates, totalling 240 fits


In [ ]:
# ── 9. Save model ────────────────────────────────────────────
import joblib
model_path = '/content/drive/MyDrive/GEE_exports/rf_biomass_model_2020.pkl'
joblib.dump(rf_best, model_path)
print(f'\nModel saved to: {model_path}')